
# CSE498R Final Thesis Notebook — Stage 2(200 images)  
**Topic:** Advanced Stronger Inference Pipeline  
**Pipeline:** YOLO Segmentation → Refined Removal Mask → LaMa Global Inpainting → Stable Diffusion Local Refinement → Residual Detection Check

This notebook is the **core architecture notebook** for your thesis.

## Main idea
We already finished:
- Stage 0: ZIP upload + auto mask generation
- Stage 1: preprocessing + filtering + context ring + split

Now we build the **stronger final inference system**:
1. load processed image, mask, and context ring
2. create a better final removal mask
3. run global LaMa inpainting
4. crop the local region around the removed object
5. run diffusion-based local inpainting refinement
6. paste refined patch back into the global result
7. run residual person detection
8. optionally refine again if a person remains

## Notes
- This notebook is for advanced inference and evaluation
- It is designed for Google Colab Pro
- The local diffusion model may be heavy, so a safe fallback path is included


In [1]:
# CELL 1 — install only once, then restart runtime

!pip uninstall -y numpy lpips basemap opencv-python opencv-python-headless opencv-contrib-python
!pip cache purge

!pip install -q numpy==1.26.4
!pip install -q ultralytics
!pip install -q simple-lama-inpainting
!pip install -q diffusers transformers accelerate safetensors
!pip install -q opencv-python-headless==4.10.0.84
!pip install -q pillow tqdm pandas matplotlib scikit-image
!pip install -q lpips==0.1.4

print("INSTALL CELL FINISHED — NOW RESTART RUNTIME MANUALLY")

Found existing installation: numpy 2.0.2
Uninstalling numpy-2.0.2:
  Successfully uninstalled numpy-2.0.2
Found existing installation: opencv-python 4.13.0.92
Uninstalling opencv-python-4.13.0.92:
  Successfully uninstalled opencv-python-4.13.0.92
Found existing installation: opencv-python-headless 4.13.0.92
Uninstalling opencv-python-headless-4.13.0.92:
  Successfully uninstalled opencv-python-headless-4.13.0.92
Found existing installation: opencv-contrib-python 4.13.0.92
Uninstalling opencv-contrib-python-4.13.0.92:
  Successfully uninstalled opencv-contrib-python-4.13.0.92
Files removed: 0
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 105.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dopamine-rl 4.1.2 requires opencv-python>=3.4.8.29, which is not 

In [5]:
# ============================================================
# CELL 2: Mount Google Drive and define paths
# ============================================================

from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path

PROJECT_ROOT = Path("/content/drive/MyDrive/CSE498R_Final_Thesis")
DATA_ROOT = PROJECT_ROOT / "data"
PROCESSED_ROOT = DATA_ROOT / "processed"
SPLIT_ROOT = DATA_ROOT / "splits"
RESULT_ROOT = PROJECT_ROOT / "results"
MODEL_ROOT = PROJECT_ROOT / "models"
VIS_ROOT = PROJECT_ROOT / "visualizations"

STAGE2_ROOT = RESULT_ROOT / "stage2_advanced_pipeline" / "200"
STAGE2_IMG_ROOT = STAGE2_ROOT / "images"
STAGE2_META_ROOT = STAGE2_ROOT / "metadata"

for p in [RESULT_ROOT, MODEL_ROOT, VIS_ROOT, STAGE2_ROOT, STAGE2_IMG_ROOT, STAGE2_META_ROOT]:
    p.mkdir(parents=True, exist_ok=True)

print("Stage 2 result paths ready.")
print("STAGE2_ROOT =", STAGE2_ROOT)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Stage 2 result paths ready.
STAGE2_ROOT = /content/drive/MyDrive/CSE498R_Final_Thesis/results/stage2_advanced_pipeline/200


In [1]:
# ============================================================
# CELL 3: Imports
# ============================================================

import time
import json
import random
import warnings
warnings.filterwarnings("ignore")

import cv2
import lpips
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

from PIL import Image
from tqdm.auto import tqdm
from skimage.metrics import peak_signal_noise_ratio, structural_similarity
from simple_lama_inpainting import SimpleLama
from ultralytics import YOLO

random.seed(42)
np.random.seed(42)
torch.manual_seed(42)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("DEVICE =", DEVICE)


Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
DEVICE = cuda


In [2]:
# ============================================================
# CELL 4: Configuration
# ============================================================

CFG = {
    # "target_split": "test", # No longer filtering by split, processing a subset of the full dataset
    "max_samples": 200, # User requested to process 200 images

    "use_context_ring": True,
    "mask_extra_dilate_px": 6,

    "local_crop_expand_ratio": 1.85,
    "local_target_size": 512,

    "detector_model": "yolov8x-seg.pt",
    "residual_conf_threshold": 0.20,
    "residual_class_name": "person",

    "positive_prompt": "realistic clean natural background, photorealistic, coherent texture, natural lighting, no person, no human, high detail",
    "negative_prompt": "person, human, face, arms, legs, body, blurry, distorted, artifacts, duplicate, unrealistic, text, watermark, logo",

    "local_inpaint_model_id": "sd2-community/stable-diffusion-2-inpainting",
    "local_num_inference_steps": 30,
    "local_guidance_scale": 7.5,
    "local_strength_fallback_to_lama": True,

    "num_preview_samples": 5
}

print(json.dumps(CFG, indent=2))

{
  "max_samples": 200,
  "use_context_ring": true,
  "mask_extra_dilate_px": 6,
  "local_crop_expand_ratio": 1.85,
  "local_target_size": 512,
  "detector_model": "yolov8x-seg.pt",
  "residual_conf_threshold": 0.2,
  "residual_class_name": "person",
  "positive_prompt": "realistic clean natural background, photorealistic, coherent texture, natural lighting, no person, no human, high detail",
  "negative_prompt": "person, human, face, arms, legs, body, blurry, distorted, artifacts, duplicate, unrealistic, text, watermark, logo",
  "local_inpaint_model_id": "sd2-community/stable-diffusion-2-inpainting",
  "local_num_inference_steps": 30,
  "local_guidance_scale": 7.5,
  "local_strength_fallback_to_lama": true,
  "num_preview_samples": 5
}



## Data files expected from previous stages

This notebook expects these files to already exist:

- `data/processed/images/*.png`
- `data/processed/masks/*.png`
- `data/processed/context_rings/*.png`
- `data/splits/split_metadata.csv`


In [6]:
# ============================================================
# CELL 5: Load split metadata
# ============================================================

split_csv = SPLIT_ROOT / "split_metadata.csv"
df = pd.read_csv(split_csv)

# The user requested to process the "complete 200 images" from the original subset.
# We will load all samples from the dataset up to CFG["max_samples"].
df_target = df.copy().reset_index(drop=True)

if CFG["max_samples"] is not None and CFG["max_samples"] > 0:
    df_target = df_target.head(CFG["max_samples"]).copy()

print("Loaded split file:", split_csv)
print(f"Samples to process: {len(df_target)}")
display(df_target.head())

Loaded split file: /content/drive/MyDrive/CSE498R_Final_Thesis/data/splits/split_metadata.csv
Samples to process: 104


,id,image_path,mask_path,ring_path,height,width,mask_ratio,bbox_x1,bbox_y1,bbox_x2,bbox_y2,adaptive_dilation,split
0,000006_000000006954,/content/drive/MyDrive/CSE498R_Final_Thesis/da...,/content/drive/MyDrive/CSE498R_Final_Thesis/da...,/content/drive/MyDrive/CSE498R_Final_Thesis/da...,960,1280,0.310260,0,208,1280,908,28,test
1,000007_000000009400,/content/drive/MyDrive/CSE498R_Final_Thesis/da...,/content/drive/MyDrive/CSE498R_Final_Thesis/da...,/content/drive/MyDrive/CSE498R_Final_Thesis/da...,960,1280,0.343040,0,98,1280,960,28,train
2,000008_000000009590,/content/drive/MyDrive/CSE498R_Final_Thesis/da...,/content/drive/MyDrive/CSE498R_Final_Thesis/da...,/content/drive/MyDrive/CSE498R_Final_Thesis/da...,854,1280,0.305782,0,124,1280,838,28,train
3,000010_000000013291,/content/drive/MyDrive/CSE498R_Final_Thesis/da...,/content/drive/MyDrive/CSE498R_Final_Thesis/da...,/content/drive/MyDrive/CSE498R_Final_Thesis/da...,858,1280,0.200045,59,175,1280,764,28,train
4,000011_000000013659,/content/drive/MyDrive/CSE498R_Final_Thesis/da...,/content/drive/MyDrive/CSE498R_Final_Thesis/da...,/content/drive/MyDrive/CSE498R_Final_Thesis/da...,960,1280,0.205387,308,14,1280,960,28,test


In [7]:
# ============================================================
# CELL 6: Load main models
# ============================================================

# 6A. Global inpainting model: LaMa
simple_lama = SimpleLama()
print("Global inpainting model (LaMa) ready.")

# 6B. Residual detector
detector = YOLO(CFG["detector_model"])
print("Residual detector ready:", CFG["detector_model"])

# 6C. Local diffusion inpainting pipeline
# Corrected loading code for diffusers inpainting pipeline.
# If loading fails, the notebook will still continue with LaMa fallback.

local_pipe = None
local_pipe_name = None

try:
    from diffusers import AutoPipelineForInpainting

    if DEVICE == "cuda":
        local_pipe = AutoPipelineForInpainting.from_pretrained(
            CFG["local_inpaint_model_id"],
            torch_dtype=torch.float16,
            variant="fp16"
        )
        local_pipe = local_pipe.to("cuda")
    else:
        local_pipe = AutoPipelineForInpainting.from_pretrained(
            CFG["local_inpaint_model_id"],
            torch_dtype=torch.float32
        )

    # Optional memory optimization
    try:
        local_pipe.enable_xformers_memory_efficient_attention()
    except Exception:
        pass

    # Optional CPU offload for memory saving in some Colab cases
    try:
        local_pipe.enable_attention_slicing()
    except Exception:
        pass

    local_pipe_name = CFG["local_inpaint_model_id"]
    print("Local diffusion pipeline loaded:", local_pipe_name)

except Exception as e:
    print("Could not load local diffusion inpainting pipeline.")
    print("Reason:", e)
    print("Fallback mode will use local LaMa refinement only.")

Downloading: "https://github.com/enesmsahin/simple-lama-inpainting/releases/download/v0.1.0/big-lama.pt" to /root/.cache/torch/hub/checkpoints/big-lama.pt
100%|██████████| 196M/196M [00:03<00:00, 55.3MB/s]


Global inpainting model (LaMa) ready.
Residual detector ready: yolov8x-seg.pt


Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.


model_index.json:   0%|          | 0.00/544 [00:00<?, ?B/s]

Fetching 13 files:   0%|          | 0/13 [00:00<?, ?it/s]

Loading pipeline components...:   0%|          | 0/6 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/372 [00:00<?, ?it/s]

CLIPTextModel LOAD REPORT from: /root/.cache/huggingface/hub/models--sd2-community--stable-diffusion-2-inpainting/snapshots/5f74973cbb64c8568780732c17f43eb269d63a0d/text_encoder
Key                                | Status     |  | 
-----------------------------------+------------+--+-
text_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Local diffusion pipeline loaded: sd2-community/stable-diffusion-2-inpainting


In [8]:
# ============================================================
# CELL 7: Metric model
# ============================================================

lpips_model = lpips.LPIPS(net='alex').to(DEVICE)
print("LPIPS model ready.")


Setting up [LPIPS] perceptual loss: trunk [alex], v[0.1], spatial [off]
Downloading: "https://download.pytorch.org/models/alexnet-owt-7be5be79.pth" to /root/.cache/torch/hub/checkpoints/alexnet-owt-7be5be79.pth



  0%|          | 0.00/233M [00:00<?, ?B/s]
  5%|▍         | 11.4M/233M [00:00<00:01, 119MB/s]
 11%|█         | 24.5M/233M [00:00<00:01, 129MB/s]
 16%|█▌        | 37.2M/233M [00:00<00:01, 131MB/s]
 22%|██▏       | 50.1M/233M [00:00<00:01, 132MB/s]
 27%|██▋       | 62.8M/233M [00:00<00:01, 125MB/s]
 32%|███▏      | 74.8M/233M [00:00<00:01, 114MB/s]
 38%|███▊      | 88.0M/233M [00:00<00:01, 122MB/s]
 43%|████▎     | 99.9M/233M [00:00<00:01, 118MB/s]
 48%|████▊     | 112M/233M [00:00<00:01, 121MB/s] 
 53%|█████▎    | 124M/233M [00:01<00:00, 123MB/s]
 59%|█████▊    | 136M/233M [00:01<00:00, 124MB/s]
 64%|██████▍   | 149M/233M [00:01<00:00, 127MB/s]
 69%|██████▉   | 162M/233M [00:01<00:00, 125MB/s]
 74%|███████▍  | 174M/233M [00:01<00:00, 120MB/s]
 79%|███████▉  | 185M/233M [00:01<00:00, 120MB/s]
 84%|████████▍ | 197M/233M [00:01<00:00, 116MB/s]
 89%|████████▉ | 208M/233M [00:01<00:00, 115MB/s]
100%|██████████| 233M/233M [00:01<00:00, 125MB/s]


Loading model from: /usr/local/lib/python3.12/dist-packages/lpips/weights/v0.1/alex.pth
LPIPS model ready.


In [9]:
def compute_region_metrics(reference_rgb, predicted_rgb, region_mask):
    """
    Safe metric computation.
    It automatically fixes shape mismatch before PSNR / SSIM / LPIPS.
    """

    region_mask = binary_mask(region_mask)

    # Make predicted image same size as reference if needed
    if predicted_rgb.shape[:2] != reference_rgb.shape[:2]:
        predicted_rgb = cv2.resize(
            predicted_rgb,
            (reference_rgb.shape[1], reference_rgb.shape[0]),
            interpolation=cv2.INTER_CUBIC
        )

    # Make mask same size as reference if needed
    if region_mask.shape[:2] != reference_rgb.shape[:2]:
        region_mask = cv2.resize(
            region_mask,
            (reference_rgb.shape[1], reference_rgb.shape[0]),
            interpolation=cv2.INTER_NEAREST
        )

    bbox = get_bbox_from_mask(region_mask)
    if bbox is None:
        return {
            "psnr_region": np.nan,
            "ssim_region": np.nan,
            "lpips_full": np.nan,
            "out_ssim": np.nan
        }

    x1, y1, x2, y2 = bbox

    ref_crop = reference_rgb[y1:y2, x1:x2]
    pred_crop = predicted_rgb[y1:y2, x1:x2]

    # Final safety check
    if ref_crop.shape[:2] != pred_crop.shape[:2]:
        pred_crop = cv2.resize(
            pred_crop,
            (ref_crop.shape[1], ref_crop.shape[0]),
            interpolation=cv2.INTER_CUBIC
        )

    # If crop too small, SSIM may fail
    if ref_crop.shape[0] < 7 or ref_crop.shape[1] < 7:
        ssim_region = np.nan
    else:
        ssim_region = structural_similarity(
            ref_crop,
            pred_crop,
            channel_axis=2,
            data_range=255
        )

    psnr_region = peak_signal_noise_ratio(ref_crop, pred_crop, data_range=255)
    lpips_full = compute_lpips(reference_rgb, predicted_rgb)

    # Full image outside/background consistency proxy
    if reference_rgb.shape[:2] != predicted_rgb.shape[:2]:
        predicted_rgb = cv2.resize(
            predicted_rgb,
            (reference_rgb.shape[1], reference_rgb.shape[0]),
            interpolation=cv2.INTER_CUBIC
        )

    if reference_rgb.shape[0] < 7 or reference_rgb.shape[1] < 7:
        out_ssim = np.nan
    else:
        out_ssim = structural_similarity(
            reference_rgb,
            predicted_rgb,
            channel_axis=2,
            data_range=255
        )

    return {
        "psnr_region": float(psnr_region),
        "ssim_region": float(ssim_region) if not np.isnan(ssim_region) else np.nan,
        "lpips_full": float(lpips_full),
        "out_ssim": float(out_ssim) if not np.isnan(out_ssim) else np.nan
    }

In [10]:
# ============================================================
# CELL 9: One-sample pipeline function
# ============================================================

def process_one_sample(row):
    image = read_rgb(row["image_path"])
    mask = read_gray(row["mask_path"])
    ring = read_gray(row["ring_path"])

    final_mask = merge_mask_and_ring(
        mask,
        ring,
        use_context_ring=CFG["use_context_ring"],
        extra_dilate_px=CFG["mask_extra_dilate_px"]
    )

    t0 = time.time()
    global_result = run_lama(image, final_mask)
    t_global = time.time() - t0

    bbox = get_bbox_from_mask(final_mask)
    if bbox is None:
        return None

    local_bbox = expand_bbox_to_square(
        bbox=bbox,
        image_shape=image.shape,
        expand_ratio=CFG["local_crop_expand_ratio"]
    )

    local_image = crop_by_bbox(global_result, local_bbox)
    local_mask = crop_by_bbox(final_mask, local_bbox)

    t1 = time.time()
    local_result, local_method = run_local_diffusion_or_fallback(local_image, local_mask)
    t_local = time.time() - t1

    fused_result = smooth_blend_patch(
        global_image=global_result,
        local_patch=local_result,
        local_mask=local_mask,
        bbox=local_bbox,
        blur_ksize=21
    )

    residual_count, residual_max_conf = detect_residual_persons(fused_result)

    second_pass_used = False
    if residual_count > 0:
        second_pass_used = True
        local_mask_2 = dilate_mask(local_mask, 8)
        local_result_2, local_method_2 = run_local_diffusion_or_fallback(local_image, local_mask_2)
        fused_result = smooth_blend_patch(
            global_image=fused_result,
            local_patch=local_result_2,
            local_mask=local_mask_2,
            bbox=local_bbox,
            blur_ksize=21
        )
        residual_count, residual_max_conf = detect_residual_persons(fused_result)
        local_method = local_method + " + second_pass"

    total_time = t_global + t_local

    metrics = compute_region_metrics(
        reference_rgb=image,
        predicted_rgb=fused_result,
        region_mask=final_mask
    )

    return {
        "id": row["id"],
        "final_mask_area_ratio": float((final_mask > 0).sum()) / float(final_mask.shape[0] * final_mask.shape[1]),
        "local_method": local_method,
        "residual_count": residual_count,
        "residual_max_conf": residual_max_conf,
        "global_time_sec": t_global,
        "local_time_sec": t_local,
        "total_time_sec": total_time,
        "psnr_region": metrics["psnr_region"],
        "ssim_region": metrics["ssim_region"],
        "lpips_full": metrics["lpips_full"],
        "out_ssim": metrics["out_ssim"],
        "image": image,
        "mask": mask,
        "ring": ring,
        "final_mask": final_mask,
        "global_result": global_result,
        "fused_result": fused_result,
        "local_bbox": local_bbox,
        "second_pass_used": second_pass_used
    }


In [14]:
# ============================================================
# CELL 8: Helper Functions (Image I/O, Mask Processing)
# ============================================================

def read_rgb(path):
    img = cv2.imread(str(path))
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    return img

def read_gray(path):
    mask = cv2.imread(str(path), cv2.IMREAD_GRAYSCALE)
    return mask

def binary_mask(mask):
    return (mask > 127).astype(np.uint8) * 255

def merge_mask_and_ring(mask, ring, use_context_ring=True, extra_dilate_px=0):
    final_mask = binary_mask(ring)
    if use_context_ring:
        ring_mask = binary_mask(ring)
        final_mask = np.maximum(final_mask, ring_mask)

    if extra_dilate_px > 0:
        kernel = np.ones((extra_dilate_px, extra_dilate_px), np.uint8)
        final_mask = cv2.dilate(final_mask, kernel, iterations=1)

    return final_mask

def get_bbox_from_mask(mask):
    if mask is None or np.max(mask) == 0:
        return None

    # Find contours in the binary mask
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    if not contours:
        return None

    # Get the bounding box for the largest contour (or all combined)
    x, y, w, h = cv2.boundingRect(np.vstack(contours))
    return (x, y, x + w, y + h)

def expand_bbox_to_square(bbox, image_shape, expand_ratio=1.0):
    x1, y1, x2, y2 = bbox
    h, w = image_shape[:2]

    # Calculate center and current size
    center_x, center_y = (x1 + x2) / 2, (y1 + y2) / 2
    current_size = max(x2 - x1, y2 - y1)

    # Calculate new size
    target_size = int(current_size * expand_ratio)
    if target_size % 2 != 0: # Ensure even for diffusion models
        target_size += 1

    # Calculate new coordinates
    new_x1 = int(center_x - target_size / 2)
    new_y1 = int(center_y - target_size / 2)
    new_x2 = int(center_x + target_size / 2)
    new_y2 = int(center_y + target_size / 2)

    # Clamp to image boundaries
    new_x1 = max(0, new_x1)
    new_y1 = max(0, new_y1)
    new_x2 = min(w, new_x2)
    new_y2 = min(h, new_y2)

    # Ensure it remains a square after clamping if possible
    # This part might need adjustment based on exact behavior desired at image borders.
    # For simplicity, we ensure minimum size here, potentially not perfect square at edges.
    actual_width = new_x2 - new_x1
    actual_height = new_y2 - new_y1

    # If one dimension is smaller after clamping, expand the other to maintain aspect ratio
    if actual_width < target_size:
        diff = target_size - actual_width
        new_x1 = max(0, new_x1 - diff // 2)
        new_x2 = min(w, new_x2 + (diff - diff // 2))
    if actual_height < target_size:
        diff = target_size - actual_height
        new_y1 = max(0, new_y1 - diff // 2)
        new_y2 = min(h, new_y2 + (diff - diff // 2))

    return (new_x1, new_y1, new_x2, new_y2)

def crop_by_bbox(image, bbox):
    x1, y1, x2, y2 = bbox
    return image[y1:y2, x1:x2]

def run_lama(image, mask):
    # LaMa expects RGB image and binary mask
    # SimpleLama handles input conversion for mask internally if it's 0-255 grayscale
    lama_result_pil = simple_lama(image, mask)
    return np.array(lama_result_pil) # Convert PIL Image to numpy array

def run_local_diffusion_or_fallback(local_image, local_mask, strength=1.0):
    if local_pipe is None or CFG["local_strength_fallback_to_lama"] is False:
        # Fallback to LaMa if diffusion model not loaded or explicitly disabled
        return simple_lama(local_image, local_mask), "lama_only"

    # Ensure local_image is PIL Image for diffusers
    pil_image = Image.fromarray(local_image)
    pil_mask = Image.fromarray(local_mask)

    # Resize to local_target_size for diffusion
    pil_image = pil_image.resize((CFG["local_target_size"], CFG["local_target_size"]), Image.LANCZOS)
    pil_mask = pil_mask.resize((CFG["local_target_size"], CFG["local_target_size"]), Image.NEAREST)

    # Convert mask to 1-channel binary for diffusion
    pil_mask_diffusion = pil_mask.convert("L")

    # Run diffusion inpainting
    try:
        output = local_pipe(
            image=pil_image,
            mask_image=pil_mask_diffusion,
            prompt=CFG["positive_prompt"],
            negative_prompt=CFG["negative_prompt"],
            num_inference_steps=CFG["local_num_inference_steps"],
            guidance_scale=CFG["local_guidance_scale"],
            strength=strength # Use default or specified strength
        ).images[0]
        return np.array(output), "diffusion_local"
    except Exception as e:
        print(f"Warning: Local diffusion failed with strength={strength}. Falling back to LaMa. Error: {e}")
        # Fallback to LaMa if diffusion fails
        return simple_lama(local_image, local_mask), "lama_fallback"

def smooth_blend_patch(global_image, local_patch, local_mask, bbox, blur_ksize=21):
    x1, y1, x2, y2 = bbox

    # Resize local_patch to match bbox size (if diffusion output is different)
    patch_w = x2 - x1
    patch_h = y2 - y1
    if local_patch.shape[:2] != (patch_h, patch_w):
        local_patch = cv2.resize(local_patch, (patch_w, patch_h), interpolation=cv2.INTER_LANCZOS4)
        local_mask = cv2.resize(local_mask, (patch_w, patch_h), interpolation=cv2.INTER_NEAREST)

    # Create a feathered mask for blending
    blend_mask = np.zeros(global_image.shape[:2], dtype=np.float32)
    blend_mask[y1:y2, x1:x2] = local_mask.astype(np.float32) / 255.0

    # Apply Gaussian blur to feather the mask edges
    blend_mask = cv2.GaussianBlur(blend_mask, (blur_ksize, blur_ksize), 0)
    blend_mask = np.expand_dims(blend_mask, axis=-1) # Add channel dimension

    # Blend the global image and the local patch
    fused_image = global_image.copy()
    fused_image[y1:y2, x1:x2] = (blend_mask[y1:y2, x1:x2] * local_patch + \
                                 (1 - blend_mask[y1:y2, x1:x2]) * global_image[y1:y2, x1:x2]).astype(np.uint8)
    return fused_image

def detect_residual_persons(image):
    results = detector(image, verbose=False)
    residual_count = 0
    residual_max_conf = 0.0

    for r in results:
        for *xyxy, conf, cls in r.boxes.data.cpu().numpy():
            if detector.names[int(cls)] == CFG["residual_class_name"] and conf > CFG["residual_conf_threshold"]:
                residual_count += 1
                if conf > residual_max_conf:
                    residual_max_conf = conf

    return residual_count, residual_max_conf

def dilate_mask(mask, ksize=5):
    kernel = np.ones((ksize, ksize), np.uint8)
    return cv2.dilate(mask, kernel, iterations=1)

def compute_lpips(img1, img2):
    # Ensure images are in the correct format (Tensor, -1 to 1 range, RGB)
    img1_tensor = lpips.im2tensor(img1.astype(np.float32) / 255.0).to(DEVICE)
    img2_tensor = lpips.im2tensor(img2.astype(np.float32) / 255.0).to(DEVICE)

    # Compute LPIPS distance
    with torch.no_grad():
        dist = lpips_model(img1_tensor, img2_tensor)
    return dist.item()

In [15]:
# ============================================================
# CELL 10: Run advanced pipeline on full 200 split
# ============================================================

all_outputs = []
result_rows = []

for _, row in tqdm(df_target.iterrows(), total=len(df_target), desc="Stage 2 inference"):
    out = process_one_sample(row)
    if out is None:
        continue

    sample_id = out["id"]

    Image.fromarray(out["image"]).save(STAGE2_IMG_ROOT / f"{sample_id}_input.png")
    Image.fromarray(out["mask"]).save(STAGE2_IMG_ROOT / f"{sample_id}_mask.png")
    Image.fromarray(out["ring"]).save(STAGE2_IMG_ROOT / f"{sample_id}_ring.png")
    Image.fromarray(out["final_mask"]).save(STAGE2_IMG_ROOT / f"{sample_id}_final_mask.png")
    Image.fromarray(out["global_result"]).save(STAGE2_IMG_ROOT / f"{sample_id}_global_lama.png")
    Image.fromarray(out["fused_result"]).save(STAGE2_IMG_ROOT / f"{sample_id}_final_result.png")

    result_rows.append({
        "id": sample_id,
        "final_mask_area_ratio": out["final_mask_area_ratio"],
        "local_method": out["local_method"],
        "residual_count": out["residual_count"],
        "residual_max_conf": out["residual_max_conf"],
        "global_time_sec": out["global_time_sec"],
        "local_time_sec": out["local_time_sec"],
        "total_time_sec": out["total_time_sec"],
        "psnr_region": out["psnr_region"],
        "ssim_region": out["ssim_region"],
        "lpips_full": out["lpips_full"],
        "out_ssim": out["out_ssim"],
        "second_pass_used": out["second_pass_used"]
    })

    all_outputs.append(out)

results_df = pd.DataFrame(result_rows)
# Update CSV filename to reflect processing all samples, not just a specific split
results_csv = STAGE2_META_ROOT / f"stage2_results_all_samples_{len(df_target)}.csv"
results_df.to_csv(results_csv, index=False)

print("Stage 2 inference complete.")
print("Saved results CSV:", results_csv)
display(results_df.head())

Stage 2 inference:   0%|          | 0/104 [00:00<?, ?it/s]

  0%|          | 0/30 [00:00<?, ?it/s]

  0%|          | 0/30 [00:00<?, ?it/s]

  0%|          | 0/30 [00:00<?, ?it/s]

  0%|          | 0/30 [00:00<?, ?it/s]

...

  0%|          | 0/30 [00:00<?, ?it/s]

  0%|          | 0/30 [00:00<?, ?it/s]

  0%|          | 0/30 [00:00<?, ?it/s]

Stage 2 inference complete.
Saved results CSV: /content/drive/MyDrive/CSE498R_Final_Thesis/results/stage2_advanced_pipeline/200/metadata/stage2_results_all_samples_104.csv


,id,final_mask_area_ratio,local_method,residual_count,residual_max_conf,global_time_sec,local_time_sec,total_time_sec,psnr_region,ssim_region,lpips_full,out_ssim,second_pass_used
0,000006_000000006954,0.167360,diffusion_local + second_pass,5,0.952842,4.755159,8.000615,12.755774,18.510262,0.789831,0.000054,0.833182,True
1,000007_000000009400,0.146906,diffusion_local + second_pass,8,0.917215,0.833251,7.410718,8.243969,24.608641,0.882575,0.000024,0.890789,True
2,000008_000000009590,0.165057,diffusion_local + second_pass,5,0.914632,0.817885,6.870336,7.688221,18.421228,0.787040,0.000060,0.810384,True
3,000010_000000013291,0.196208,diffusion_local + second_pass,4,0.930241,3.715023,6.819200,10.534223,15.289417,0.484244,0.000102,0.533843,True
4,000011_000000013659,0.127910,diffusion_local + second_pass,5,0.931922,0.842062,7.097706,7.939767,19.271819,0.854918,0.000040,0.886366,True


## Implementing Synthetic Evaluation

This section outlines how to perform evaluation using a synthetic dataset, where a true clean-background ground truth image is available for direct comparison.

In [11]:
import copy
from pathlib import Path # Import Path
import json # Import json for json.dumps

# Re-define PROJECT_ROOT, DATA_ROOT, SPLIT_ROOT, RESULT_ROOT for this cell's scope if it hasn't been run
PROJECT_ROOT = Path("/content/drive/MyDrive/CSE498R_Final_Thesis")
DATA_ROOT = PROJECT_ROOT / "data"
SPLIT_ROOT = DATA_ROOT / "splits"
RESULT_ROOT = PROJECT_ROOT / "results"

# Extend the configuration for synthetic evaluation
SYNTH_CFG = copy.deepcopy(CFG)
SYNTH_CFG.update({
    "synth_split_metadata_csv": str(SPLIT_ROOT / "synthetic_split_metadata.csv"), # Path to your synthetic split CSV
    "synth_results_root": str(RESULT_ROOT / "stage2_advanced_pipeline" / "synthetic"),
    "synth_max_samples": 10, # Number of synthetic samples to process
    "synth_img_root": None, # Override if synthetic images are in a different base folder
    "synth_mask_root": None, # Override if synthetic masks are in a different base folder
    "synth_ring_root": None, # Override if synthetic context rings are in a different base folder
    "synth_gt_root": None # Base path for ground truth images
})

# Ensure synthetic result paths exist
Path(SYNTH_CFG["synth_results_root"]).mkdir(parents=True, exist_ok=True);
(Path(SYNTH_CFG["synth_results_root"]) / "images").mkdir(parents=True, exist_ok=True);
(Path(SYNTH_CFG["synth_results_root"]) / "metadata").mkdir(parents=True, exist_ok=True)

print("Synthetic evaluation configuration ready.")
print(json.dumps(SYNTH_CFG, indent=2))

Synthetic evaluation configuration ready.
{
  "max_samples": 200,
  "use_context_ring": true,
  "mask_extra_dilate_px": 6,
  "local_crop_expand_ratio": 1.85,
  "local_target_size": 512,
  "detector_model": "yolov8x-seg.pt",
  "residual_conf_threshold": 0.2,
  "residual_class_name": "person",
  "positive_prompt": "realistic clean natural background, photorealistic, coherent texture, natural lighting, no person, no human, high detail",
  "negative_prompt": "person, human, face, arms, legs, body, blurry, distorted, artifacts, duplicate, unrealistic, text, watermark, logo",
  "local_inpaint_model_id": "sd2-community/stable-diffusion-2-inpainting",
  "local_num_inference_steps": 30,
  "local_guidance_scale": 7.5,
  "local_strength_fallback_to_lama": true,
  "num_preview_samples": 5,
  "synth_split_metadata_csv": "/content/drive/MyDrive/CSE498R_Final_Thesis/data/splits/synthetic_split_metadata.csv",
  "synth_results_root": "/content/drive/MyDrive/CSE498R_Final_Thesis/results/stage2_advanced_

In [31]:
# ============================================================
# New: One-sample pipeline function for Synthetic Data
# ============================================================

def process_one_synthetic_sample(row):
    # Add checks for file existence before reading
    image_path = Path(row["image_path"])
    mask_path = Path(row["mask_path"])
    ring_path = Path(row["ring_path"])
    ground_truth_path = Path(row["ground_truth_path"])

    if not image_path.exists():
        print(f"Warning: Image not found for ID {row['id']} at {image_path}. Skipping sample.")
        return None
    if not mask_path.exists():
        print(f"Warning: Mask not found for ID {row['id']} at {mask_path}. Skipping sample.")
        return None
    if not ring_path.exists():
        print(f"Warning: Ring not found for ID {row['id']} at {ring_path}. Skipping sample.")
        return None
    if not ground_truth_path.exists():
        print(f"Warning: Ground truth not found for ID {row['id']} at {ground_truth_path}. Skipping sample.")
        return None

    image = read_rgb(image_path)
    mask = read_gray(mask_path)
    ring = read_gray(ring_path)
    ground_truth_image = read_rgb(ground_truth_path) # Load ground truth

    final_mask = merge_mask_and_ring(
        mask,
        ring,
        use_context_ring=SYNTH_CFG["use_context_ring"],
        extra_dilate_px=SYNTH_CFG["mask_extra_dilate_px"]
    )

    t0 = time.time()
    global_result = run_lama(image, final_mask)
    t_global = time.time() - t0

    bbox = get_bbox_from_mask(final_mask)
    if bbox is None:
        return None

    local_bbox = expand_bbox_to_square(
        bbox=bbox,
        image_shape=image.shape,
        expand_ratio=SYNTH_CFG["local_crop_expand_ratio"]
    )

    local_image = crop_by_bbox(global_result, local_bbox)
    local_mask = crop_by_bbox(final_mask, local_bbox)

    t1 = time.time()
    local_result, local_method = run_local_diffusion_or_fallback(local_image, local_mask)
    t_local = time.time() - t1

    fused_result = smooth_blend_patch(
        global_image=global_result,
        local_patch=local_result,
        local_mask=local_mask,
        bbox=local_bbox,
        blur_ksize=21
    )

    residual_count, residual_max_conf = detect_residual_persons(fused_result)

    second_pass_used = False
    if residual_count > 0:
        second_pass_used = True
        local_mask_2 = dilate_mask(local_mask, 8)
        local_result_2, local_method_2 = run_local_diffusion_or_fallback(local_image, local_mask_2)
        fused_result = smooth_blend_patch(
            global_image=fused_result,
            local_patch=local_result_2,
            local_mask=local_mask_2,
            bbox=local_bbox,
            blur_ksize=21
        )
        residual_count, residual_max_conf = detect_residual_persons(fused_result)
        local_method = local_method + " + second_pass"

    total_time = t_global + t_local

    # Crucial change: compute metrics against the ground truth image
    metrics = compute_region_metrics(
        reference_rgb=ground_truth_image,
        predicted_rgb=fused_result,
        region_mask=final_mask # Use the same final mask for region metrics
    )

    return {
        "id": row["id"],
        "final_mask_area_ratio": float((final_mask > 0).sum()) / float(final_mask.shape[0] * final_mask.shape[1]),
        "local_method": local_method,
        "residual_count": residual_count,
        "residual_max_conf": residual_max_conf,
        "global_time_sec": t_global,
        "local_time_sec": t_local,
        "total_time_sec": total_time,
        "psnr_region": metrics["psnr_region"],
        "ssim_region": metrics["ssim_region"],
        "lpips_full": metrics["lpips_full"], # LPIPS with GT as reference
        "out_ssim": metrics["out_ssim"], # Out SSIM with GT as reference
        "image": image,
        "mask": mask,
        "ring": ring,
        "final_mask": final_mask,
        "global_result": global_result,
        "fused_result": fused_result,
        "local_bbox": local_bbox,
        "ground_truth_image": ground_truth_image,
        "second_pass_used": second_pass_used
    }

In [36]:
from pathlib import Path

# 1. Define SYNTHETIC_DATA_BASE_PATH
SYNTHETIC_DATA_BASE_PATH = PROJECT_ROOT / "data" / "synthetic_processed"

# 2. Create the base directory if it does not already exist
SYNTHETIC_DATA_BASE_PATH.mkdir(parents=True, exist_ok=True)
print(f"Ensured base synthetic data directory exists: {SYNTHETIC_DATA_BASE_PATH}")

# 3. Create the subdirectories
subdirectories = ["images", "masks", "context_rings", "ground_truths"]
for subdir in subdirectories:
    (SYNTHETIC_DATA_BASE_PATH / subdir).mkdir(parents=True, exist_ok=True)
    print(f"Ensured subdirectory exists: {SYNTHETIC_DATA_BASE_PATH / subdir}")

print("All synthetic data directories are ready.")

Ensured base synthetic data directory exists: /content/drive/MyDrive/CSE498R_Final_Thesis/data/synthetic_processed
Ensured subdirectory exists: /content/drive/MyDrive/CSE498R_Final_Thesis/data/synthetic_processed/images
Ensured subdirectory exists: /content/drive/MyDrive/CSE498R_Final_Thesis/data/synthetic_processed/masks
Ensured subdirectory exists: /content/drive/MyDrive/CSE498R_Final_Thesis/data/synthetic_processed/context_rings
Ensured subdirectory exists: /content/drive/MyDrive/CSE498R_Final_Thesis/data/synthetic_processed/ground_truths
All synthetic data directories are ready.


In [13]:
import cv2
import numpy as np
from PIL import Image
from tqdm.auto import tqdm

print("Generating synthetic data and ground truths...")

# Ensure simple_lama is initialized (from CELL 6)
if 'simple_lama' not in locals():
    print("Initializing SimpleLama for synthetic data generation...")
    simple_lama = SimpleLama()

# Define paths for saving synthetic components
SYNTH_IMG_SAVE_ROOT = SYNTHETIC_DATA_BASE_PATH / "images"
SYNTH_MASK_SAVE_ROOT = SYNTHETIC_DATA_BASE_PATH / "masks"
SYNTH_RING_SAVE_ROOT = SYNTHETIC_DATA_BASE_PATH / "context_rings"
SYNTH_GT_SAVE_ROOT = SYNTHETIC_DATA_BASE_PATH / "ground_truths"

synth_data_rows_for_csv = []

for index, row in tqdm(df_target.iterrows(), total=len(df_target), desc="Generating Synthetic Ground Truths"):
    sample_id = row['id']

    # Read original components (image, mask, ring)
    original_image = read_rgb(Path(row["image_path"]))
    original_mask = read_gray(Path(row["mask_path"]))
    original_ring = read_gray(Path(row["ring_path"]))

    # Generate the final mask to be used for LaMa inpainting to create GT
    # Use the same parameters as the main pipeline for consistency
    final_mask_for_gt = merge_mask_and_ring(
        original_mask,
        original_ring,
        use_context_ring=CFG["use_context_ring"],
        extra_dilate_px=CFG["mask_extra_dilate_px"]
    )

    # Create synthetic ground truth image using LaMa
    # The ground truth is the original image with the person (and context ring) removed
    ground_truth_image = run_lama(original_image, final_mask_for_gt)

    # Save the original image, mask, ring, and generated ground truth
    Image.fromarray(original_image).save(SYNTH_IMG_SAVE_ROOT / f"{sample_id}_image.png")
    Image.fromarray(original_mask).save(SYNTH_MASK_SAVE_ROOT / f"{sample_id}_mask.png")
    Image.fromarray(original_ring).save(SYNTH_RING_SAVE_ROOT / f"{sample_id}_ring.png")
    Image.fromarray(ground_truth_image).save(SYNTH_GT_SAVE_ROOT / f"{sample_id}_ground_truth.png")

    # Collect data for synthetic_split_metadata.csv
    synth_data_rows_for_csv.append({
        "id": sample_id,
        "image_path": str(SYNTH_IMG_SAVE_ROOT / f"{sample_id}_image.png"),
        "mask_path": str(SYNTH_MASK_SAVE_ROOT / f"{sample_id}_mask.png"),
        "ring_path": str(SYNTH_RING_SAVE_ROOT / f"{sample_id}_ring.png"),
        "height": row['height'],
        "width": row['width'],
        "mask_ratio": row['mask_ratio'],
        "bbox_x1": row['bbox_x1'],
        "bbox_y1": row['bbox_y1'],
        "bbox_x2": row['bbox_x2'],
        "bbox_y2": row['bbox_y2'],
        "adaptive_dilation": row['adaptive_dilation'],
        "split": row['split'],
        "ground_truth_path": str(SYNTH_GT_SAVE_ROOT / f"{sample_id}_ground_truth.png")
    })

# Create and save the synthetic_split_metadata.csv
df_synthetic_metadata = pd.DataFrame(synth_data_rows_for_csv)
synthetic_split_csv_path = SPLIT_ROOT / "synthetic_split_metadata.csv"
df_synthetic_metadata.to_csv(synthetic_split_csv_path, index=False)

print(f"Synthetic ground truths and metadata generated and saved to: {synthetic_split_csv_path}")
display(df_synthetic_metadata.head())

Generating synthetic data and ground truths...
Initializing SimpleLama for synthetic data generation...


NameError: name 'SimpleLama' is not defined

In [6]:
from pathlib import Path
from PIL import Image # Re-import Image explicitly for clarity, though it might be global
from tqdm.auto import tqdm # Re-import tqdm explicitly for clarity, though it might be global

# ============================================================
# New: Run advanced pipeline on Synthetic split
# ============================================================

all_synth_outputs = []
synth_result_rows = []

synth_img_save_root = Path(SYNTH_CFG["synth_results_root"]) / "images"
synth_meta_save_root = Path(SYNTH_CFG["synth_results_root"]) / "metadata"

for _, row in tqdm(df_synth.iterrows(), total=len(df_synth), desc="Stage 2 Synthetic inference"):
    out = process_one_synthetic_sample(row)
    if out is None:
        continue

    sample_id = out["id"]

    # Save synthetic inputs and outputs
    Image.fromarray(out["image"]).save(synth_img_save_root / f"{sample_id}_input.png")
    Image.fromarray(out["mask"]).save(synth_img_save_root / f"{sample_id}_mask.png")
    Image.fromarray(out["ring"]).save(synth_img_save_root / f"{sample_id}_ring.png")
    Image.fromarray(out["final_mask"]).save(synth_img_save_root / f"{sample_id}_final_mask.png")
    Image.fromarray(out["global_result"]).save(synth_img_save_root / f"{sample_id}_global_lama.png")
    Image.fromarray(out["fused_result"]).save(synth_img_save_root / f"{sample_id}_final_result.png")
    Image.fromarray(out["ground_truth_image"]).save(synth_img_save_root / f"{sample_id}_ground_truth.png")

    synth_result_rows.append({
        "id": sample_id,
        "final_mask_area_ratio": out["final_mask_area_ratio"],
        "local_method": out["local_method"],
        "residual_count": out["residual_count"],
        "residual_max_conf": out["residual_max_conf"],
        "global_time_sec": out["global_time_sec"],
        "local_time_sec": out["local_time_sec"],
        "total_time_sec": out["total_time_sec"],
        "psnr_region": out["psnr_region"],
        "ssim_region": out["ssim_region"],
        "lpips_full": out["lpips_full"],
        "out_ssim": out["out_ssim"],
        "second_pass_used": out["second_pass_used"]
    })

    all_synth_outputs.append(out) # Moved inside the loop

synth_results_df = pd.DataFrame(synth_result_rows)
synth_results_csv = synth_meta_save_root / f"stage2_synthetic_results_{len(df_synth)}.csv"
synth_results_df.to_csv(synth_results_csv, index=False)

print("Stage 2 Synthetic inference complete.")
print("Saved synthetic results CSV:", synth_results_csv)
display(synth_results_df.head())


NameError: name 'SYNTH_CFG' is not defined

In [ ]:
# ============================================================
# New: Summary table for Synthetic Evaluation
# ============================================================

if len(synth_results_df) == 0:
    print("No Synthetic Stage 2 results found.")
else:
    synth_summary = synth_results_df[[
        "final_mask_area_ratio",
        "residual_count",
        "residual_max_conf",
        "global_time_sec",
        "local_time_sec",
        "total_time_sec",
        "psnr_region",
        "ssim_region",
        "lpips_full",
        "out_ssim"
    ]].describe()

    print("Synthetic pipeline summary:")
    display(synth_summary)

    print("Synthetic Mean values:")
    display(synth_results_df.mean(numeric_only=True).to_frame("mean").T)

In [ ]:
# ============================================================
# CELL 11: Summary table
# ============================================================

if len(results_df) == 0:
    raise ValueError("No Stage 2 results found.")

summary = results_df[[
    "final_mask_area_ratio",
    "residual_count",
    "residual_max_conf",
    "global_time_sec",
    "local_time_sec",
    "total_time_sec",
    "psnr_region",
    "ssim_region",
    "lpips_full",
    "out_ssim"
]].describe()

print("Advanced pipeline summary:")
display(summary)

print("Mean values:")
display(results_df.mean(numeric_only=True).to_frame("mean").T)


---

In [ ]:
# ============================================================
# CELL 12: Visual preview of outputs (as usual will use for stage2 improve)
# ============================================================

num_show = min(CFG["num_preview_samples"], len(all_outputs))

for i in range(num_show):
    out = all_outputs[i]

    plt.figure(figsize=(22, 10))

    plt.subplot(2, 3, 1)
    plt.imshow(out["image"])
    plt.title(f"Input\n{out['id']}")
    plt.axis("off")

    plt.subplot(2, 3, 2)
    plt.imshow(out["final_mask"], cmap="gray")
    plt.title("Final Removal Mask")
    plt.axis("off")

    plt.subplot(2, 3, 3)
    plt.imshow(out["global_result"])
    plt.title("Global LaMa Result")
    plt.axis("off")

    plt.subplot(2, 3, 4)
    plt.imshow(out["fused_result"])
    plt.title("Final Fused Result")
    plt.axis("off")

    overlay = out["image"].copy()
    overlay[out["final_mask"] > 0] = [255, 0, 0]
    plt.subplot(2, 3, 5)
    plt.imshow(overlay)
    plt.title("Input + Final Mask Overlay")
    plt.axis("off")

    x1, y1, x2, y2 = out["local_bbox"]
    local_vis = out["fused_result"].copy()
    cv2.rectangle(local_vis, (x1, y1), (x2, y2), (0, 255, 0), 3)
    plt.subplot(2, 3, 6)
    plt.imshow(local_vis)
    plt.title(f"Local Box\nmethod={out['local_method']} | residual={out['residual_count']} | conf={out['residual_max_conf']:.3f}")
    plt.axis("off")

    plt.show()


In [ ]:
# ============================================================
# NEW CELL: Save existing results to a custom folder
# ============================================================

from pathlib import Path
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm

# Define the new custom root folder
CUSTOM_SAVE_ROOT = PROJECT_ROOT / "200 evaluation"
CUSTOM_IMG_ROOT = CUSTOM_SAVE_ROOT / "images"
CUSTOM_META_ROOT = CUSTOM_SAVE_ROOT / "metadata"

# Create the directories if they don't exist
for p in [CUSTOM_SAVE_ROOT, CUSTOM_IMG_ROOT, CUSTOM_META_ROOT]:
    p.mkdir(parents=True, exist_ok=True)

print(f"Custom save directories created under: {CUSTOM_SAVE_ROOT}")

# Save the results_df to the new metadata folder
custom_results_csv_path = CUSTOM_META_ROOT / f"stage2_results_all_samples_{len(results_df)}.csv"
results_df.to_csv(custom_results_csv_path, index=False)

print(f"Results DataFrame saved to: {custom_results_csv_path}")

# Save all necessary images from all_outputs to the new image folder
print("Saving all necessary images to custom folder...")
for out in tqdm(all_outputs, total=len(all_outputs), desc="Saving custom images"):
    sample_id = out["id"]
    Image.fromarray(out["image"]).save(CUSTOM_IMG_ROOT / f"{sample_id}_input.png")
    Image.fromarray(out["mask"]).save(CUSTOM_IMG_ROOT / f"{sample_id}_mask.png")
    Image.fromarray(out["ring"]).save(CUSTOM_IMG_ROOT / f"{sample_id}_ring.png")
    Image.fromarray(out["final_mask"]).save(CUSTOM_IMG_ROOT / f"{sample_id}_final_mask.png")
    Image.fromarray(out["global_result"]).save(CUSTOM_IMG_ROOT / f"{sample_id}_global_lama.png")
    Image.fromarray(out["fused_result"]).save(CUSTOM_IMG_ROOT / f"{sample_id}_final_result.png")

print("All requested results and images saved to the custom folder successfully!")

# Task
The previous request was stopped. I will ignore the previous plan and focus on the current task.

The plan involves defining `CFG` and `SYNTH_CFG`, loading synthetic split metadata, ensuring the processing function is ready, running the synthetic evaluation pipeline, and reviewing the results.

Based on the notebook state, it appears that `CFG` was not defined before `SYNTH_CFG` was attempted to be defined, leading to a `NameError`.

Here's the step-by-step execution:

1. **Execute cell `86784a1b`**: This cell defines the main `CFG` dictionary.
2. **Execute cell `3aa288c7`**: This cell defines the `SYNTH_CFG` dictionary, copying from `CFG` and setting `synth_max_samples` to 10.
3. **Create a new cell and execute the following code**: This step will load the synthetic split metadata into `df_synth`, respecting the `SYNTH_CFG['synth_max_samples']` value of 10.
   ```python
   import pandas as pd
   from pathlib import Path

   # Load synthetic split metadata
   synth_split_csv = Path(SYNTH_CFG["synth_split_metadata_csv"])
   df_synth = pd.read_csv(synth_split_csv)

   # Apply synth_max_samples if defined
   if SYNTH_CFG["synth_max_samples"] is not None and SYNTH_CFG["synth_max_samples"] > 0:
       df_synth = df_synth.head(SYNTH_CFG["synth_max_samples"]).copy()

   print("Loaded synthetic split file:", synth_split_csv)
   print(f"Synthetic samples to process: {len(df_synth)}")
   display(df_synth.head())
   ```
4. **Execute cell `3562ad5f`**: This cell ensures the `process_one_synthetic_sample` function, which uses `SYNTH_CFG` and `ground_truth_path` for metrics, is loaded and up-to-date.
5. **Execute cell `aa9ed01e`**: This cell will run the synthetic evaluation pipeline for the first 10 synthetic samples using `SYNTH_CFG` and `df_synth`.
6. **Execute cell `40e2d9f5`**: This cell will display the summary table of the synthetic evaluation for the processed samples.

## Define all Configurations

### Subtask:
Define both the main `CFG` and the `SYNTH_CFG` dictionaries. This is crucial for subsequent steps that rely on these configurations.


**Reasoning**:
The first instruction is to execute cell `86784a1b` to define the main `CFG` dictionary. I will provide the content of this cell as a `code_block`.



In [9]:
import json
# ============================================================
# CELL 4: Configuration
# ============================================================

CFG = {
    # "target_split": "test", # No longer filtering by split, processing a subset of the full dataset
    "max_samples": 200, # User requested to process 200 images

    "use_context_ring": True,
    "mask_extra_dilate_px": 6,

    "local_crop_expand_ratio": 1.85,
    "local_target_size": 512,

    "detector_model": "yolov8x-seg.pt",
    "residual_conf_threshold": 0.20,
    "residual_class_name": "person",

    "positive_prompt": "realistic clean natural background, photorealistic, coherent texture, natural lighting, no person, no human, high detail",
    "negative_prompt": "person, human, face, arms, legs, body, blurry, distorted, artifacts, duplicate, unrealistic, text, watermark, logo",

    "local_inpaint_model_id": "sd2-community/stable-diffusion-2-inpainting",
    "local_num_inference_steps": 30,
    "local_guidance_scale": 7.5,
    "local_strength_fallback_to_lama": True,

    "num_preview_samples": 5
}

print(json.dumps(CFG, indent=2))

{
  "max_samples": 200,
  "use_context_ring": true,
  "mask_extra_dilate_px": 6,
  "local_crop_expand_ratio": 1.85,
  "local_target_size": 512,
  "detector_model": "yolov8x-seg.pt",
  "residual_conf_threshold": 0.2,
  "residual_class_name": "person",
  "positive_prompt": "realistic clean natural background, photorealistic, coherent texture, natural lighting, no person, no human, high detail",
  "negative_prompt": "person, human, face, arms, legs, body, blurry, distorted, artifacts, duplicate, unrealistic, text, watermark, logo",
  "local_inpaint_model_id": "sd2-community/stable-diffusion-2-inpainting",
  "local_num_inference_steps": 30,
  "local_guidance_scale": 7.5,
  "local_strength_fallback_to_lama": true,
  "num_preview_samples": 5
}
